# Is it normal? One 118-second checkout breaks the line

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Normal Probability Plot and Normality Tests** · a worked example from the StatNex teaching guide

**The story.** A cafe manager recorded four things for 20 customers: the weight of each coffee, the time at the till, the minute of the hour they walked in, and how much they spent. Many statistical tests assume data look like a bell curve, so she checks each column first.

**The question.** Which of these four measurements look normally distributed, and which do not?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/normal_probability_plot.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Cup weight (g),Checkout time (sec),Arrival minute,Spend per customer ($)
1.0,356.0,49.0,14.0,2.79
2.0,357.4,51.0,39.0,9.31
3.0,351.8,39.0,52.0,9.03
4.0,346.5,47.0,8.0,4.51
5.0,335.4,51.0,9.0,9.28
6.0,361.1,48.0,29.0,10.25
7.0,353.3,47.0,17.0,3.91
8.0,353.5,45.0,48.0,33.3
9.0,350.1,40.0,51.0,32.52
10.0,346.0,56.0,16.0,6.04
11.0,357.1,44.0,42.0,7.41
12.0,349.3,118.0,26.0,7.42
13.0,352.2,48.0,2.0,7.38
14.0,356.3,38.0,3.0,5.82
15.0,363.7,43.0,21.0,4.81
16.0,349.8,43.0,0.0,4.25
17.0,342.6,44.0,26.0,7.27
18.0,358.9,52.0,56.0,33.72
19.0,349.8,46.0,48.0,6.22
20.0,348.0,51.0,52.0,1.76

These cells save `lilliefors.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile lilliefors.py
"""Lilliefors' P value for the Kolmogorov-Smirnov test of normality.

When the normal's mean and SD are estimated from the same data, the fitted curve hugs
the sample, D runs small, and the ordinary Kolmogorov P value is far too large. This
is the published approximation to Lilliefors' distribution that R's
nortest::lillie.test uses: Dallal & Wilkinson (1986) for P <= 0.1, and above that
Stephens' modified statistic D * (sqrt(n) - 0.01 + 0.85 / sqrt(n)) mapped through four
polynomials (D'Agostino & Stephens 1986, Table 4.7). statsmodels' lilliefors(...,
pvalmethod="approx") agrees below 0.1 but reads a table above it.
"""

import math


def lilliefors_p(d, n):
    """P value of the KS statistic d from n observations, mean and SD estimated."""
    # Dallal-Wilkinson: n above 100 is handled by rescaling d to n = 100.
    if n <= 100:
        dd, m = d, n
    else:
        dd, m = d * (n / 100) ** 0.49, 100
    p = math.exp(-7.01256 * dd * dd * (m + 2.78019) + 2.99587 * dd * math.sqrt(m + 2.78019)
                 - 0.122119 + 0.974598 / math.sqrt(m) + 1.67997 / m)
    if p <= 0.1:
        return p
    # Stephens: the modified statistic through the polynomial for its range.
    k = (math.sqrt(n) - 0.01 + 0.85 / math.sqrt(n)) * d
    if k <= 0.302:
        return 1.0
    if k <= 0.5:
        p = 2.76773 - 19.828315 * k + 80.709644 * k**2 - 138.55152 * k**3 + 81.218052 * k**4
    elif k <= 0.9:
        p = -4.901232 + 40.662806 * k - 97.490286 * k**2 + 94.029866 * k**3 - 32.355711 * k**4
    elif k <= 1.31:
        p = 6.198765 - 19.558097 * k + 23.186922 * k**2 - 12.234627 * k**3 + 2.423045 * k**4
    else:
        p = 0.0
    return min(max(p, 0.0), 1.0)

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from lilliefors import lilliefors_p

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column; blank and text cells are left out. The tests need at least 8 values.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Checkout time (sec)"
x = pd.to_numeric(data[column], errors="coerce").dropna().to_numpy(float)
n = x.size
mean, sd = np.mean(x), np.std(x, ddof=1)

## Descriptive statistics

In [ ]:
# Skewness and kurtosis here are the plain moment ratios m3 / m2^1.5 and m4 / m2²
# (0 and 3 for a normal distribution) -- the forms D'Agostino's tests are built on,
# not the sample-corrected SKEW and KURT that Descriptive Statistics reports.
# Skewness away from 0 means a lopsided shape; kurtosis above 3 means heavy tails or
# an outlier, below 3 a flat shape.
summary = pd.Series({"n": n, "mean": mean, "variance": np.var(x, ddof=1), "sd": sd,
                     "min": x.min(), "max": x.max(), "skewness": stats.skew(x),
                     "kurtosis": stats.kurtosis(x, fisher=False)})
print(summary.to_string())

## D'Agostino's tests of normality

In [ ]:
# Skewness and kurtosis are each turned into a z statistic with a two-sided P
# (SciPy's skewtest, D'Agostino's, and kurtosistest, Anscombe & Glynn's); normaltest,
# the omnibus K² test, adds their squares and compares the sum with chi-square on
# 2 df. Each asks whether the shape is further from a normal one than sampling alone
# would explain. A small P is evidence against normality; a large P does not show the
# data are normal, and with 20 or so values these tests miss sizeable departures.
skew_test = stats.skewtest(x)
kurt_test = stats.kurtosistest(x)
omnibus = stats.normaltest(x)
dagostino = pd.DataFrame(
    [[skew_test.statistic, np.nan, skew_test.pvalue],
     [kurt_test.statistic, np.nan, kurt_test.pvalue],
     [omnibus.statistic, 2, omnibus.pvalue]],
    index=["skewness (z)", "kurtosis (z)", "omnibus (chi-square)"], columns=["statistic", "df", "P"],
)
print()
print(dagostino)

## Goodness-of-fit tests of normality

In [ ]:
# Kolmogorov-Smirnov against a normal with the sample's own mean and SD. Because those
# were estimated from the same data, the plain KS P value is far too large, so
# Lilliefors' correction is used (lilliefors.py, the algorithm of R's
# nortest::lillie.test; statsmodels' lilliefors() reads a table above P = 0.1 and so
# differs slightly there). Shapiro-Wilk compares the sorted data with the values a
# normal sample would give; it is SciPy's (Royston's 1995 algorithm) and usually the
# most powerful of these tests for small samples. Read P as for the tests above.
ks = stats.kstest(x, stats.norm(mean, sd).cdf)
shapiro = stats.shapiro(x)
gof = pd.DataFrame(
    [[ks.statistic, n, lilliefors_p(ks.statistic, n)],
     [shapiro.statistic, n, shapiro.pvalue]],
    index=["Kolmogorov-Smirnov (Lilliefors)", "Shapiro-Wilk"], columns=["statistic", "n", "P"],
)
print()
print(gof)

## Normal probability plot

In [ ]:
# Each value's plotting position is (rank - 0.5) / n (tied values share their average
# rank), turned into the z a normal distribution would put there. The rows stay in
# the data's order.
p_observed = (stats.rankdata(x) - 0.5) / n
z_expected = stats.norm.ppf(p_observed)
z_observed = (x - mean) / sd
plot_data = pd.DataFrame({
    "observed": x, "expected": mean + sd * z_expected,
    "z observed": z_observed, "z expected": z_expected,
    "p observed": p_observed, "p expected": stats.norm.cdf(z_observed),
})
print()
print(plot_data.to_string(index=False))

# Points on the straight line mean the data look normal. A bow means skew, an S-shape
# a flat or heavy-tailed shape, and a lone point far off the line an outlier.
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.scatter(z_expected, x)
line = np.array([z_expected.min(), z_expected.max()])
ax.plot(line, mean + sd * line)
ax.set_xlabel("Standardised expected value")
ax.set_ylabel(column)
fig.savefig("figures/normal_probability_plot.png", dpi=150)
plt.close(fig)

## What the result says

A normal distribution is the bell curve that many tests assume. The checkout times range from 38 to 118 seconds, but 118 is one customer alone. That single value makes the skewness (a longer tail on one side) 3.61 and the kurtosis (heavy tails) 15.4. Both are far too high for a bell curve.

Every normality test agrees. The Shapiro-Wilk statistic is only 0.497, where a value close to 1 means a good fit, and all the P values are practically zero. A P value is the chance of data this far from a bell curve if they really came from one.

So checkout time is not normal, and the reason is one extreme value. In the plot, 19 points sit together and one point lies far away.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.